# Archetype 2 — Base + PV

## Purpose

Computes the annual Total Cost of Electricity (TCoE) for **Archetype 2: Base + PV** (10 kWp rooftop PV, no storage or flexible devices) across all 3 operational strategies and 7 DSOs → **21 runs**.

Because there is no flexible device, the schedule is deterministic for all strategies:
- $E^t_{grid} = \max(0,\; E^t_{base} - E^t_{PV})$ — net grid draw after PV self-consumption
- $E^t_{feedin} = \max(0,\; E^t_{PV} - E^t_{base})$ — surplus PV exported at spot price (Direktvermarktung)

All three strategies therefore produce **identical schedules**; only the DSO bill differs per run.

## Input files (`inputs/`)

| File | Content |
|---|---|
| `base_demand_h25_4500kwh_2026_15min.csv` | 35,040-slot BDEW H25 demand (kWh/slot) |
| `pv_kassel_10kwp_2026_15min.csv` | 10 kWp PV generation Kassel 2026 (kWh/slot) |
| `spot_prices_de_lu_2025_15min.csv` | Day-ahead spot price (ct/kWh, 15-min) |
| `dso_tariffs_residential_2026.csv` | Residential base tariffs for 7 DSOs |
| `residential_taxes_2026.csv` | German levies (ct/kWh, pre-VAT) |

## Output (`outputs/`)

`results_base_pv_2026.csv` — 21 rows × cost-component columns.

## Billing convention

- Net spot cost = grid purchases − feed-in revenue, both valued at the 15-min spot price.
- Levies (pre-VAT) are included in the net subtotal; a single 19% VAT is applied to the full subtotal.
- `cost_smart_operating_net_eur = 100 €/year` (flat iMSys / metering lump-sum, net).
- No §14a rebate (no registered controllable device).
- `annual_energy_kWh` in results = annual grid draw (not site consumption).

## Thesis reference

Chapter 5, Section 5.1 — Results: Archetype 2 (Base + PV)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ── Path configuration ─────────────────────────────────────────────────────────
def find_repo_root(marker='README.md'):
    p = Path('__file__').resolve().parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'Repo root (containing {marker}) not found.')

REPO_ROOT = find_repo_root()
INPUTS    = REPO_ROOT / 'inputs'
OUTPUTS   = REPO_ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

print(f'Repo root : {REPO_ROOT}')
print(f'Inputs    : {INPUTS}')
print(f'Outputs   : {OUTPUTS}')

Repo root : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub
Inputs    : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/inputs
Outputs   : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/outputs


## Step 1 — Load inputs

In [2]:
demand = pd.read_csv(INPUTS / 'base_demand_h25_4500kwh_2026_15min.csv',
                     parse_dates=['timestamp'])
pv     = pd.read_csv(INPUTS / 'pv_kassel_10kwp_2026_15min.csv',
                     parse_dates=['datetime'])
spot   = pd.read_csv(INPUTS / 'spot_prices_de_lu_2025_15min.csv',
                     parse_dates=['timestamp'])

# Align on a common timestamp index
pv = pv.rename(columns={'datetime': 'timestamp'})
df = demand.merge(pv[['timestamp', 'energy_kWh']], on='timestamp', how='inner')
df = df.merge(spot, on='timestamp', how='inner')
df = df.rename(columns={'demand_kwh': 'E_base_kWh',
                         'energy_kWh': 'E_pv_kWh',
                         'price_ct_kWh': 'c_spot_ct_kWh'})

# Schedule: deterministic for all strategies (no flexible device)
df['E_grid_kWh']   = (df['E_base_kWh'] - df['E_pv_kWh']).clip(lower=0)
df['E_feedin_kWh'] = (df['E_pv_kWh']  - df['E_base_kWh']).clip(lower=0)

# Tariff data
dso_tariffs = pd.read_csv(INPUTS / 'dso_tariffs_residential_2026.csv')
DSO_LIST    = list(dso_tariffs['DSO'])

taxes = pd.read_csv(INPUTS / 'residential_taxes_2026.csv')
tax_row = taxes.loc[taxes['region'] == 'DE'].iloc[0]
TAX_PRE_VAT_CT_KWH = float(tax_row['Total_no_VAT_ct_kWh'])  # applied to grid kWh; VAT added via subtotal

VAT_RATE   = 0.19
SMART_OP   = 100.0   # flat net iMSys / smart-operating lump-sum (EUR/year)
ARCHETYPE  = 2
STRATEGIES = ['no_flex', 'dt_flex', 'tcoe_flex']

annual_grid_kWh   = df['E_grid_kWh'].sum()
annual_feedin_kWh = df['E_feedin_kWh'].sum()
print(f'Annual site consumption : {df["E_base_kWh"].sum():.2f} kWh')
print(f'Annual grid draw        : {annual_grid_kWh:.2f} kWh')
print(f'Annual feed-in          : {annual_feedin_kWh:.2f} kWh')
print(f'Levies pre-VAT          : {TAX_PRE_VAT_CT_KWH} ct/kWh (on grid draw)')

Annual site consumption : 4500.00 kWh
Annual grid draw        : 2664.24 kWh
Annual feed-in          : 9913.49 kWh
Levies pre-VAT          : 6.64 ct/kWh (on grid draw)


## Step 2 — Compute billing (21 runs)

$$\text{TCoE} = \underbrace{(C_{spot,net} + C_{DSO,fix} + C_{DSO,vol} + C_{smart} + C_{levies,pre-VAT})}_{\text{net subtotal}} \times 1.19$$

where $C_{spot,net} = \sum_t (E^t_{grid} - E^t_{feedin}) \cdot c^t_{spot}$ and levies apply to grid draw only.

In [3]:
# Pre-compute DSO-independent terms
c_spot_net      = ((df['E_grid_kWh'] - df['E_feedin_kWh']) * df['c_spot_ct_kWh']).sum() / 100.0
c_levies_pre_vat = annual_grid_kWh * TAX_PRE_VAT_CT_KWH / 100.0

records = []
run_id  = 0
for strategy in STRATEGIES:
    for dso_id in DSO_LIST:
        run_id += 1
        dso = dso_tariffs.loc[dso_tariffs['DSO'] == dso_id].iloc[0]

        c_fix_net = float(dso['Grundpreis_EUR_year'])
        c_vol_net = annual_grid_kWh * float(dso['Arbeitspreis_ct_kWh']) / 100.0

        subtotal_net = c_spot_net + c_fix_net + c_vol_net + SMART_OP + c_levies_pre_vat
        vat          = VAT_RATE * subtotal_net
        total_tcoe   = subtotal_net + vat

        records.append({
            'run_id'                      : run_id,
            'scenario_id'                 : f'{strategy}_{ARCHETYPE}_{dso_id}_2026',
            'strategy'                    : strategy,
            'household_archetype'         : ARCHETYPE,
            'dso_id'                      : dso_id,
            'annual_energy_kWh'           : round(annual_grid_kWh, 2),
            'cost_spot_net_eur'           : round(c_spot_net, 2),
            'cost_dso_fixed_net_eur'      : round(c_fix_net, 2),
            'cost_dso_volumetric_net_eur' : round(c_vol_net, 2),
            'cost_smart_operating_net_eur': round(SMART_OP, 2),
            'cost_levies_pre_vat_eur'     : round(c_levies_pre_vat, 2),
            'subtotal_net_eur'            : round(subtotal_net, 2),
            'vat_eur'                     : round(vat, 2),
            'total_tcoe_eur'              : round(total_tcoe, 2),
        })

results = pd.DataFrame(records)
results

,run_id,scenario_id,strategy,household_archetype,dso_id,annual_energy_kWh,cost_spot_net_eur,cost_dso_fixed_net_eur,cost_dso_volumetric_net_eur,cost_smart_operating_net_eur,cost_levies_pre_vat_eur,subtotal_net_eur,vat_eur,total_tcoe_eur
0,1,no_flex_2_Westnetz_2026,no_flex,2,Westnetz,2664.24,-276.65,80.30,253.90,100.0,176.91,334.46,63.55,398.00
1,2,no_flex_2_Bayernwerk_2026,no_flex,2,Bayernwerk,2664.24,-276.65,98.55,125.75,100.0,176.91,224.56,42.67,267.22
2,3,no_flex_2_E.DIS_2026,no_flex,2,E.DIS,2664.24,-276.65,76.65,145.73,100.0,176.91,222.64,42.30,264.94
3,4,no_flex_2_Netze BW_2026,no_flex,2,Netze BW,2664.24,-276.65,84.00,201.68,100.0,176.91,285.94,54.33,340.26
4,5,no_flex_2_Stromnetz Berlin_2026,no_flex,2,Stromnetz Berlin,2664.24,-276.65,33.36,198.75,100.0,176.91,232.37,44.15,276.52
5,6,no_flex_2_SH Netz_2026,no_flex,2,SH Netz,2664.24,-276.65,94.90,170.51,100.0,176.91,265.66,50.48,316.14
6,7,no_flex_2_MITNETZ STROM_2026,no_flex,2,MITNETZ STROM,2664.24,-276.65,73.00,168.11,100.0,176.91,241.37,45.86,287.23
7,8,dt_flex_2_Westnetz_2026,dt_flex,2,Westnetz,2664.24,-276.65,80.30,253.90,100.0,176.91,334.46,63.55,398.00
8,9,dt_flex_2_Bayernwerk_2026,dt_flex,2,Bayernwerk,2664.24,-276.65,98.55,125.75,100.0,176.91,224.56,42.67,267.22
9,10,dt_flex_2_E.DIS_2026,dt_flex,2,E.DIS,2664.24,-276.65,76.65,145.73,100.0,176.91,222.64,42.30,264.94


## Step 3 — Summary

In [4]:
# no_flex only (strategies are identical for Archetype 2)
summary = (results[results['strategy'] == 'no_flex']
           [['dso_id', 'cost_spot_net_eur', 'cost_dso_fixed_net_eur',
             'cost_dso_volumetric_net_eur', 'cost_levies_pre_vat_eur',
             'vat_eur', 'total_tcoe_eur']]
           .set_index('dso_id')
           .rename(columns={
               'cost_spot_net_eur'           : 'Spot net (€)',
               'cost_dso_fixed_net_eur'       : 'DSO fix (€)',
               'cost_dso_volumetric_net_eur'  : 'DSO vol (€)',
               'cost_levies_pre_vat_eur'      : 'Levies pre-VAT (€)',
               'vat_eur'                      : 'VAT (€)',
               'total_tcoe_eur'               : 'TCoE (€/yr)',
           }))

print(f'Annual grid draw  : {annual_grid_kWh:.2f} kWh  (site: 4,500 kWh)')
print(f'Annual feed-in    : {annual_feedin_kWh:.2f} kWh  (valued at spot)')
print(f'TCoE range        : {results["total_tcoe_eur"].min()} – {results["total_tcoe_eur"].max()} EUR/year')
print(f'(all strategies identical for Archetype 2)\n')
summary

Annual grid draw  : 2664.24 kWh  (site: 4,500 kWh)
Annual feed-in    : 9913.49 kWh  (valued at spot)
TCoE range        : 264.94 – 398.0 EUR/year
(all strategies identical for Archetype 2)



,Spot net (€),DSO fix (€),DSO vol (€),Levies pre-VAT (€),VAT (€),TCoE (€/yr)
dso_id,,,,,,
Westnetz,-276.65,80.30,253.90,176.91,63.55,398.00
Bayernwerk,-276.65,98.55,125.75,176.91,42.67,267.22
E.DIS,-276.65,76.65,145.73,176.91,42.30,264.94
Netze BW,-276.65,84.00,201.68,176.91,54.33,340.26
Stromnetz Berlin,-276.65,33.36,198.75,176.91,44.15,276.52
SH Netz,-276.65,94.90,170.51,176.91,50.48,316.14
MITNETZ STROM,-276.65,73.00,168.11,176.91,45.86,287.23


## Step 4 — Export

In [5]:
out = OUTPUTS / 'results_base_pv_2026.csv'
results.to_csv(out, index=False)
print(f'Saved : {out}')
print(f'Rows  : {len(results)}  (3 strategies × 7 DSOs)')

Saved : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/outputs/results_base_pv_2026.csv
Rows  : 21  (3 strategies × 7 DSOs)
